cdc simply track what change occur on our source table so that it only read those changes not entire table

delta table is only the source for cdc , beacause cdc is a feature of delta table which is not provide by others like readind data from files or others.

manual cdc and create_auto_cdc flow are same but there used cases different:

    - used manual cdc when we already have a Delta table and changes happen inside it

    - used auto cdc when we receive change events from outside systems, we are NOT updating table directly
    👉 we are just appending events

In [0]:
%sql
USE CATALOG DEMO;
DROP SCHEMA IF EXISTS CDC_FEED;
CREATE SCHEMA CDC_FEED;
USE SCHEMA CDC_FEED;
DROP TABLE IF EXISTS bronze_customers;

CREATE TABLE bronze_customers (
  id INT,
  name STRING,
  city STRING
)
TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql
INSERT INTO bronze_customers VALUES (1, 'John', 'NY'), (2, 'Jane', 'LA'), (3, 'Bob','IND');

In [0]:
%sql
-- # STEP 3: Make Changes (CDC Events)
-- # 👉 Update

UPDATE bronze_customers
SET city = 'Bangalore'
WHERE id = 2;

-- # 👉 Delete

DELETE FROM bronze_customers
WHERE id = 3;

-- # 👉 Insert

INSERT INTO bronze_customers VALUES
(4, 'D', 'Chennai');

In [0]:
# 🔹 STEP 4: Read Change Data Feed (CDC Output)

cdc_df = spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 0) \
    .table("bronze_customers")

cdc_df.display()

In [0]:
%sql
-- # 🔹 STEP 5: Create Silver Table (Target)

DROP TABLE IF EXISTS silver_customers;

CREATE TABLE silver_customers (
  id INT,
  name STRING,
  city STRING
);

In [0]:
# STEP 6: Apply CDC using MERGE (Incremental Load)

from pyspark.sql.functions import col

# basically here we are read only those record which are change not entire table
cdc_df = spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 0) \
    .table("bronze_customers")

# Only keep latest changes (ignore preimage)
cdc_filtered = cdc_df.filter(
    col("_change_type").isin("insert", "update_postimage", "delete")
)

cdc_filtered.createOrReplaceTempView("cdc_view")

spark.sql("""
MERGE INTO silver_customers AS target
USING cdc_view AS source
ON target.id = source.id

WHEN MATCHED AND source._change_type = 'delete' THEN DELETE

WHEN MATCHED AND source._change_type = 'update_postimage' THEN
  UPDATE SET *

WHEN NOT MATCHED AND source._change_type = 'insert' THEN
  INSERT *
""")

In [0]:
%sql
-- 🔹 STEP 7: Check Final Silver Table

SELECT * FROM silver_customers;